# Visualización de Métricas de Evaluación XAI

Histogramas de distribución de AggDiv e IXD por algoritmo.
Cada algoritmo se superpone en la misma gráfica con un color distinto.

**Estructura esperada de ficheros:**
```
output/metricas_evaluacion_muestra/
    evaluacion_{algoritmo}_AggDiv_{timestamp}.csv
    evaluacion_{algoritmo}_IXD_{timestamp}.csv
    evaluacion_{algoritmo}_MIL_{timestamp}.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict

# ── Configuración estética ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#f9f9f7',
    'axes.grid':         True,
    'grid.color':        'white',
    'grid.linewidth':    1.2,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
})

In [ ]:
# ── Rutas ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent.parent   # ajusta si ejecutas desde otra carpeta
EVAL_DIR     = PROJECT_ROOT / 'output' / 'metricas_evaluacion_muestra'
OUTPUT_DIR   = PROJECT_ROOT / 'output' / 'visualizacion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Leyendo CSVs desde: {EVAL_DIR}')
print(f'Guardando figuras en: {OUTPUT_DIR}')

In [ ]:
# ── Cargar todos los CSVs y agrupar por métrica ──────────────────────────────
#
# Estructura resultante:
#   datos[metrica][algoritmo] = pd.DataFrame
#
# Detecta automáticamente qué métricas y algoritmos están disponibles.

METRICAS_USUARIO = ['AggDiv', 'IXD']   # granularidad usuario — tienen columna 'usuario'
METRICAS_SISTEMA = ['MIL']             # granularidad sistema — una fila por algoritmo

datos: dict = defaultdict(dict)        # datos[metrica][algoritmo] = df

for csv_path in sorted(EVAL_DIR.glob('evaluacion_*.csv')):
    stem  = csv_path.stem              # ej: evaluacion_kg_jaccard_similarity_AggDiv_20260404
    partes = stem.split('_')

    # Detectar métrica (última parte antes del timestamp)
    metrica     = None
    idx_metrica = None
    for m in METRICAS_USUARIO + METRICAS_SISTEMA:
        if f'_{m}_' in stem:
            metrica     = m
            idx_metrica = stem.index(f'_{m}_') + 1
            break

    if metrica is None:
        print(f'  ⚠️  No se reconoce métrica en: {csv_path.name}')
        continue

    # Nombre del algoritmo: todo lo que hay entre 'evaluacion_' y '_{metrica}_'
    prefijo     = 'evaluacion_'
    sufijo      = f'_{metrica}_'
    algoritmo   = stem[len(prefijo) : stem.index(sufijo)]

    df = pd.read_csv(csv_path)
    datos[metrica][algoritmo] = df
    print(f'  ✅  {metrica:8s}  {algoritmo:40s}  ({len(df)} filas)')

print(f'\nMétricas detectadas : {list(datos.keys())}')
for m, algs in datos.items():
    print(f'  {m}: {list(algs.keys())}')

In [ ]:
# ── Paleta de colores por algoritmo ─────────────────────────────────────────
#
# Se asigna automáticamente según el orden de aparición.
# KG en tonos azules/morados, CF en tonos cálidos — se detecta por prefijo.

COLORES_KG = ['#3B5BDB', '#7048E8', '#1C7ED6', '#0CA678']
COLORES_CF = ['#E03131', '#D6336C', '#E8590C', '#F59F00']

def asignar_colores(algoritmos: list) -> dict:
    colores = {}
    idx_kg, idx_cf = 0, 0
    for alg in sorted(algoritmos):
        if alg.startswith('kg_'):
            colores[alg] = COLORES_KG[idx_kg % len(COLORES_KG)]
            idx_kg += 1
        else:
            colores[alg] = COLORES_CF[idx_cf % len(COLORES_CF)]
            idx_cf += 1
    return colores

# Pre-calcular colores globales para que sean consistentes entre gráficas
todos_algoritmos = set()
for algs in datos.values():
    todos_algoritmos.update(algs.keys())

COLORES = asignar_colores(todos_algoritmos)
print('Colores asignados:')
for alg, col in COLORES.items():
    print(f'  {alg:40s}  {col}')

In [ ]:
# ── Función principal: histograma superpuesto ────────────────────────────────

def plot_histograma_metrica(
    metrica:    str,
    columna:    str,
    datos_algs: dict,
    colores:    dict,
    titulo:     str  = None,
    xlabel:     str  = None,
    bins:       int  = 15,
    kde:        bool = True,
    guardar:    Path = None,
):
    """
    Histograma superpuesto de `columna` para todos los algoritmos en datos_algs.

    - Barras semitransparentes (alpha=0.35) para ver la superposición.
    - Curva KDE suavizada encima para leer la forma de la distribución.
    - Línea vertical de la media de cada algoritmo.
    """
    from scipy.stats import gaussian_kde

    fig, ax = plt.subplots(figsize=(10, 5))

    # Rango común para todos los algoritmos
    todos_vals = []
    for df in datos_algs.values():
        if columna in df.columns:
            todos_vals.extend(df[columna].dropna().tolist())
    if not todos_vals:
        print(f'  ⚠️  Sin datos para {metrica} / {columna}')
        return

    x_min = min(todos_vals)
    x_max = max(todos_vals)
    # Pequeño margen
    rango  = x_max - x_min if x_max > x_min else 1
    x_min -= rango * 0.05
    x_max += rango * 0.05

    handles = []

    for algoritmo in sorted(datos_algs.keys()):
        df  = datos_algs[algoritmo]
        if columna not in df.columns:
            continue
        vals = df[columna].dropna().values
        if len(vals) == 0:
            continue

        color = colores.get(algoritmo, '#888888')
        label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')

        # Histograma semitransparente
        ax.hist(
            vals,
            bins=bins,
            range=(x_min, x_max),
            color=color,
            alpha=0.30,
            edgecolor='white',
            linewidth=0.6,
        )

        # Curva KDE
        if kde and len(np.unique(vals)) > 1:
            kde_func  = gaussian_kde(vals, bw_method='scott')
            x_kde     = np.linspace(x_min, x_max, 300)
            y_kde     = kde_func(x_kde)
            # Escalar la KDE al mismo eje que el histograma
            n_vals    = len(vals)
            bin_width = (x_max - x_min) / bins
            y_kde_sc  = y_kde * n_vals * bin_width
            ax.plot(x_kde, y_kde_sc, color=color, linewidth=2.2, label=label)
        else:
            # Sin KDE (todos los valores iguales): solo la leyenda
            ax.axvline(vals[0], color=color, linewidth=2, label=label)

        # Línea de la media
        media = np.mean(vals)
        ax.axvline(
            media,
            color=color,
            linewidth=1.2,
            linestyle='--',
            alpha=0.8,
        )

        handles.append(mpatches.Patch(color=color, alpha=0.7, label=label))

    ax.set_xlabel(xlabel or columna, fontsize=12)
    ax.set_ylabel('# usuarios', fontsize=12)
    ax.set_title(titulo or f'Distribución de {columna}', fontsize=13, fontweight='bold', pad=12)
    ax.legend(
        handles=handles,
        fontsize=9,
        framealpha=0.85,
        loc='upper right',
        ncol=2 if len(handles) > 4 else 1,
    )
    ax.set_xlim(x_min, x_max)

    plt.tight_layout()

    if guardar:
        fig.savefig(guardar, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {guardar.name}')

    plt.show()
    plt.close(fig)

## AggDiv — Diversidad Agregada

Número de explicadores únicos que recibe cada usuario (unión sobre todas sus recomendaciones).  
Se grafica `AggDiv` (total) y cada variante `@k`.

In [ ]:
if 'AggDiv' in datos:
    # Detectar columnas disponibles (AggDiv, AggDiv@1, AggDiv@3, ...)
    ejemplo_df  = next(iter(datos['AggDiv'].values()))
    cols_aggdiv = [c for c in ejemplo_df.columns if c.startswith('AggDiv')]
    print(f'Columnas AggDiv disponibles: {cols_aggdiv}')

    for col in cols_aggdiv:
        plot_histograma_metrica(
            metrica    = 'AggDiv',
            columna    = col,
            datos_algs = datos['AggDiv'],
            colores    = COLORES,
            titulo     = f'AggDiv — Distribución de {col} por algoritmo',
            xlabel     = f'{col}  (nº explicadores únicos)',
            bins       = 12,
            kde        = True,
            guardar    = OUTPUT_DIR / f'hist_AggDiv_{col}.png',
        )
else:
    print('No hay datos de AggDiv cargados.')

## IXD — Inter-eXplanation Diversity

Diversidad entre las listas de explicadores de distintas recomendaciones del mismo usuario.  
Rango [0, 1]. NaN si el usuario solo tiene una recomendación.

In [ ]:
if 'IXD' in datos:
    ejemplo_df = next(iter(datos['IXD'].values()))
    cols_ixd   = [c for c in ejemplo_df.columns if c.startswith('IXD')]
    print(f'Columnas IXD disponibles: {cols_ixd}')

    for col in cols_ixd:
        plot_histograma_metrica(
            metrica    = 'IXD',
            columna    = col,
            datos_algs = datos['IXD'],
            colores    = COLORES,
            titulo     = f'IXD — Distribución de {col} por algoritmo',
            xlabel     = f'{col}  (0 = mismos explicadores, 1 = totalmente distintos)',
            bins       = 12,
            kde        = True,
            guardar    = OUTPUT_DIR / f'hist_IXD_{col}.png',
        )
else:
    print('No hay datos de IXD cargados.')

## MIL — Mean Inter-List Diversity

Métrica de sistema: un único valor por algoritmo.  
Se representa como gráfica de barras horizontal (no histograma, porque no hay distribución).

In [ ]:
if 'MIL' in datos:
    # Construir tabla resumen: algoritmo × variante MIL
    filas = []
    for algoritmo, df in datos['MIL'].items():
        fila = {'algoritmo': algoritmo}
        cols_mil = [c for c in df.columns if c.startswith('MIL')]
        for col in cols_mil:
            fila[col] = df[col].iloc[0]
        filas.append(fila)

    df_mil = pd.DataFrame(filas).set_index('algoritmo')
    print(df_mil.to_string())

    # Gráfica de barras horizontal para MIL (es un escalar, no distribución)
    cols_mil = [c for c in df_mil.columns if c.startswith('MIL')]

    for col in cols_mil:
        fig, ax = plt.subplots(figsize=(9, max(3, len(df_mil) * 0.55)))
        algoritmos = df_mil.index.tolist()
        valores    = df_mil[col].values
        colores_barras = [COLORES.get(a, '#888') for a in algoritmos]

        bars = ax.barh(algoritmos, valores, color=colores_barras, alpha=0.8, height=0.5)
        ax.set_xlabel(f'{col}  (0 = sin personalización, 1 = totalmente personalizado)', fontsize=11)
        ax.set_title(f'MIL — {col} por algoritmo (métrica de sistema)', fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1.05)
        ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
        ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        ruta = OUTPUT_DIR / f'barras_MIL_{col}.png'
        fig.savefig(ruta, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {ruta.name}')
        plt.show()
        plt.close(fig)
else:
    print('No hay datos de MIL cargados.')

## Vista global — AggDiv e IXD en un mismo panel

Comparativa rápida de las dos métricas principales lado a lado para la variante sin @k.

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pares = [
    ('AggDiv', 'AggDiv', axes[0], 'AggDiv  (nº explicadores únicos)'),
    ('IXD',    'IXD',    axes[1], 'IXD  (0 = iguales, 1 = distintos)'),
]

for metrica, columna, ax, xlabel in pares:
    if metrica not in datos:
        ax.set_title(f'{metrica}: sin datos')
        continue

    todos_vals = []
    for df in datos[metrica].values():
        if columna in df.columns:
            todos_vals.extend(df[columna].dropna().tolist())

    if not todos_vals:
        ax.set_title(f'{columna}: sin datos')
        continue

    x_min = min(todos_vals)
    x_max = max(todos_vals)
    rango  = x_max - x_min if x_max > x_min else 1
    x_min -= rango * 0.05
    x_max += rango * 0.05

    handles = []
    for algoritmo in sorted(datos[metrica].keys()):
        df   = datos[metrica][algoritmo]
        if columna not in df.columns:
            continue
        vals = df[columna].dropna().values
        if len(vals) == 0:
            continue

        color = COLORES.get(algoritmo, '#888')
        label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')

        ax.hist(vals, bins=12, range=(x_min, x_max),
                color=color, alpha=0.28, edgecolor='white', linewidth=0.5)

        if len(np.unique(vals)) > 1:
            kde_f     = gaussian_kde(vals, bw_method='scott')
            x_kde     = np.linspace(x_min, x_max, 300)
            y_kde     = kde_f(x_kde) * len(vals) * (x_max - x_min) / 12
            ax.plot(x_kde, y_kde, color=color, linewidth=2)

        ax.axvline(np.mean(vals), color=color, linewidth=1.1, linestyle='--', alpha=0.75)
        handles.append(mpatches.Patch(color=color, alpha=0.75, label=label))

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('# usuarios', fontsize=10)
    ax.set_title(columna, fontsize=12, fontweight='bold')
    ax.legend(handles=handles, fontsize=8, framealpha=0.85,
              ncol=2 if len(handles) > 4 else 1)
    ax.set_xlim(x_min, x_max)
    ax.set_facecolor('#f9f9f7')
    ax.grid(True, color='white', linewidth=1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle('Distribución de métricas XAI por algoritmo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
ruta_global = OUTPUT_DIR / 'hist_global_AggDiv_IXD.png'
fig.savefig(ruta_global, dpi=150, bbox_inches='tight')
print(f'💾 Guardado: {ruta_global.name}')
plt.show()
plt.close(fig)